# Phase 2 — Multi-Task Circuit Sharing: Addition + Subtraction + Multiplication

**Research question (Phase 2):** Does adding multiplication mod p — which is algebraically unrelated to add/sub — break the shared circuit found in Phase 1, or does the model somehow accommodate all three operations in a single shared representation?

## Why multiplication is a harder test
- Addition and subtraction are related by a negation mod p: `a − b = a + (p−b)`. The Fourier circuit for addition naturally handles subtraction with minimal modification.
- Multiplication mod p is structurally different. For prime p, it connects to the discrete logarithm problem and does **not** decompose as a simple clockface rotation. Prior work (Nanda et al. 2023) found that multiplication uses a fundamentally different algorithmic strategy.
- If the model must solve all three tasks, it faces a genuine architectural tension: reuse the same Fourier circuit (which may not support multiplication), or build a second parallel circuit.

## What this notebook does
1. **Setup** — install dependencies (same as Phase 1, but with the `use_attn_result=True` model fix)
2. **Data** — generate a combined add+sub+mul dataset (mod 113, three op-tokens)
3. **Train** — run grokking loop, track per-task accuracy for all three operations
4. **Grokking comparison** — do all three tasks grok? Do they grok at the same epoch or different ones?
5. **Head ablation sweep** — 4 heads × 3 tasks; look for shared vs task-specific heads
6. **Activation patching** — 6 directed pairs (add↔sub, add↔mul, sub↔mul); quantify pairwise circuit overlap
7. **Fourier analysis** — compare dominant frequencies to Phase 0 (1 task) and Phase 1 (2 tasks)
8. **Cross-phase comparison** — side-by-side summary vs Phase 1 findings

## Success criteria
| Step | How you know it worked |
|------|------------------------|
| Data | `vocab_size=117`, ~11520 train, ~26769 test (30/70 split of 113²×3=38307 examples) |
| Training | All three per-task test accuracies reach ≥95% |
| Grokking | If mul grokks at a *different* epoch than add/sub → evidence of separate learning dynamics |
| Ablation | If a head hurts add+sub but not mul (or vice versa) → task-specific head for mul |
| Patching | add↔sub overlap should be HIGH (as in Phase 1); add↔mul and sub↔mul overlap should be LOWER |
| Fourier | More dominant frequencies than Phase 1? Or same? Tells you if the circuit expanded |

---
**Notebook version:** Phase 2 (add + sub + mul)  
**Model:** 1-layer, 4-head, d_model=128, mod 113, `use_attn_result=True` (fixes Phase 1 ablation bug)  
**Compute:** ~20 min on free Colab T4

## Cell 1 — Install Dependencies

Run once → Runtime → Restart session → skip to Cell 2.

In [ ]:
# Run ONCE then Runtime -> Restart session -> start from Cell 2
# Conflict warnings about gradio/diffusers are pre-existing Colab issues; ignore them.
import subprocess, sys
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformer_lens>=1.0,<3', 'einops', 'tqdm',
])
print('DONE. Now Runtime -> Restart session, then run from Cell 2.')

## Cell 2 — Imports & Reproducibility

In [ ]:
import os, random, math
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm.auto import trange
from itertools import combinations

from transformer_lens import HookedTransformer, HookedTransformerConfig
from importlib.metadata import version as _pkg_version
try:
    _tl_ver = _pkg_version('transformer_lens')
except Exception:
    import transformer_lens as _tl
    _tl_ver = getattr(_tl, '__version__', 'unknown')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Imports OK | transformer_lens {_tl_ver} | device: {DEVICE}')

## Cell 3 — Hyperparameters

In [ ]:
# Dataset
P          = 113
OPS        = ('add', 'sub', 'mul')   # Phase 2: three operations
TRAIN_FRAC = 0.3

# Model
D_MODEL  = 128
N_HEADS  = 4
D_MLP    = 512
N_LAYERS = 1
SEQ_LEN  = 4

# Training
EPOCHS           = 15_000
LR               = 1e-3
WEIGHT_DECAY     = 1.0
CHECKPOINT_EVERY = 200

os.makedirs('results', exist_ok=True)

print(f'Hyperparameters set: P={P}, ops={OPS}')
print(f'Model: d={D_MODEL}, heads={N_HEADS}, mlp={D_MLP}')
print(f'Train: epochs={EPOCHS}, lr={LR}, wd={WEIGHT_DECAY}')

## Cell 4 — Data Generation

Three operations → three op-tokens. Token layout:
- `[0, 112]` — numbers 0..112
- `113` — add token  
- `114` — sub token  
- `115` — mul token  
- `116` — `=` token

**Success check:** `vocab_size=117`, ~11520 train examples, ~26787 test examples.

In [ ]:
def make_dataset(p=113, ops=('add', 'sub', 'mul'), train_frac=0.3, seed=0):
    OP_FNS = {
        'add': lambda a, b, p: (a + b) % p,
        'sub': lambda a, b, p: (a - b) % p,
        'mul': lambda a, b, p: (a * b) % p,
    }
    rng = np.random.RandomState(seed)
    op_list = list(ops)
    equals_token = p + len(op_list)

    all_inputs, all_labels = [], []
    for op_idx, op_name in enumerate(op_list):
        op_token = p + op_idx
        fn = OP_FNS[op_name]
        for a in range(p):
            for b in range(p):
                all_inputs.append([a, op_token, b, equals_token])
                all_labels.append(fn(a, b, p))

    inputs = torch.tensor(all_inputs, dtype=torch.long)
    labels = torch.tensor(all_labels, dtype=torch.long)

    n = len(inputs)
    perm = rng.permutation(n)
    n_train = int(train_frac * n)
    train_data = (inputs[perm[:n_train]], labels[perm[:n_train]])
    test_data  = (inputs[perm[n_train:]], labels[perm[n_train:]])

    vocab_size = p + len(op_list) + 1
    return train_data, test_data, vocab_size, equals_token, op_list


train_data, test_data, VOCAB_SIZE, EQUALS_TOKEN, OP_LIST = make_dataset(
    p=P, ops=OPS, train_frac=TRAIN_FRAC, seed=SEED
)
OP_TOKENS = {op: P + i for i, op in enumerate(OP_LIST)}

print(f'Dataset ready: vocab_size={VOCAB_SIZE}, equals_token={EQUALS_TOKEN}')
print(f'Op tokens: {OP_TOKENS}')
print(f'Train: {train_data[0].shape[0]:,} | Test: {test_data[0].shape[0]:,}')

## Cell 5 — Build Model

**Key fix from Phase 1:** `use_attn_result=True` is now set in the config. This tells TransformerLens to materialise the per-head output tensor and expose it at `hook_result`. Without this, `hook_result` exists as a named hook point but is **not** actually computed separately — zeroing it has no effect (explains why Phase 1 ablation showed all-zero drops).

In [ ]:
def build_model(vocab_size, d_model=128, n_heads=4, d_mlp=512, n_layers=1,
                seq_len=4, device='cuda', seed=0):
    cfg = HookedTransformerConfig(
        n_layers=n_layers,
        d_model=d_model,
        d_head=d_model // n_heads,
        n_heads=n_heads,
        d_mlp=d_mlp,
        d_vocab=vocab_size,
        n_ctx=seq_len,
        act_fn='relu',
        normalization_type=None,
        use_attn_result=True,   # <-- Critical: enables hook_result for per-head ablation
        seed=seed,
        device=device,
    )
    return HookedTransformer(cfg)


model = build_model(
    vocab_size=VOCAB_SIZE, d_model=D_MODEL, n_heads=N_HEADS, d_mlp=D_MLP,
    n_layers=N_LAYERS, seq_len=SEQ_LEN, device=DEVICE, seed=SEED
)
n_params = sum(p.numel() for p in model.parameters())
print(f'Model built: {n_params:,} parameters')
# Verify hook_result is now in the cache
dummy = torch.zeros(1, SEQ_LEN, dtype=torch.long).to(DEVICE)
_, cache = model.run_with_cache(dummy)
hook_ok = 'blocks.0.attn.hook_result' in cache
print(f'hook_result available: {hook_ok}  <- must be True for ablation to work')

## Cell 6 — Per-Task Accuracy Helper

In [ ]:
@torch.no_grad()
def per_task_accuracy(model, inputs, labels, op_tokens_map, device=DEVICE):
    model.eval()
    inputs, labels = inputs.to(device), labels.to(device)
    logits = model(inputs)[:, -1, :]
    preds  = logits.argmax(-1)
    return {
        op: (preds[inputs[:, 1] == tok] == labels[inputs[:, 1] == tok]).float().mean().item()
        if (inputs[:, 1] == tok).sum() > 0 else float('nan')
        for op, tok in op_tokens_map.items()
    }

print('per_task_accuracy() defined.')

## Cell 7 — Training Loop

**Expected runtime:** ~18–22 minutes on a free Colab T4 (dataset is 1.5× larger than Phase 1).

**Watch for:**
- Add and sub should grok around the same epoch (as in Phase 1, ~6000–8000)
- **Multiplication may grok at a very different epoch** — possibly much later, possibly never within 15k epochs
- If mul never grokks, try `WEIGHT_DECAY=1.5` or `EPOCHS=25000` in Cell 3 and retrain

In [ ]:
train_inputs, train_labels = [x.to(DEVICE) for x in train_data]
test_inputs,  test_labels  = [x.to(DEVICE) for x in test_data]

optimizer = torch.optim.AdamW(
    model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.98)
)

history = {'epoch': [], 'train_loss': [], 'test_loss': [], 'train_acc': [], 'test_acc': []}
for op in OP_LIST:
    history[f'train_acc_{op}'] = []
    history[f'test_acc_{op}']  = []

checkpoints = []

for epoch in trange(EPOCHS, desc='Training'):
    model.train()
    logits = model(train_inputs)[:, -1, :]
    loss   = F.cross_entropy(logits, train_labels)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    if epoch % CHECKPOINT_EVERY == 0 or epoch == EPOCHS - 1:
        model.eval()
        with torch.no_grad():
            tr_l = F.cross_entropy(model(train_inputs)[:, -1, :], train_labels).item()
            te_l = F.cross_entropy(model(test_inputs)[:, -1, :],  test_labels).item()
            tr_a = (model(train_inputs)[:, -1, :].argmax(-1) == train_labels).float().mean().item()
            te_a = (model(test_inputs)[:, -1, :].argmax(-1)  == test_labels).float().mean().item()

        tr_task = per_task_accuracy(model, train_inputs, train_labels, OP_TOKENS)
        te_task = per_task_accuracy(model, test_inputs,  test_labels,  OP_TOKENS)

        history['epoch'].append(epoch)
        history['train_loss'].append(tr_l); history['test_loss'].append(te_l)
        history['train_acc'].append(tr_a);  history['test_acc'].append(te_a)
        for op in OP_LIST:
            history[f'train_acc_{op}'].append(tr_task[op])
            history[f'test_acc_{op}'].append(te_task[op])

        checkpoints.append({
            'epoch': epoch,
            'state': {k: v.cpu().clone() for k, v in model.state_dict().items()}
        })

print('\nTraining complete.')
for op in OP_LIST:
    acc = history[f'test_acc_{op}'][-1]
    grok_ep = next((ep for ep, a in zip(history['epoch'], history[f'test_acc_{op}']) if a > 0.95), 'N/A')
    status = 'GROKKED' if acc >= 0.95 else 'DID NOT GROK'
    print(f'  {op}: final test acc = {acc:.4f}  {status}  (grok epoch: {grok_ep})')

## Cell 8 — Grokking Curves (3-task comparison)

**What to look for:**
- Do all three tasks grok? At the same epoch or different ones?
- **Same epoch for add+sub (as in Phase 1), different epoch for mul** → mul uses a separate circuit that needs more time to form
- **All three at same epoch** → a single shared circuit accommodates all three (surprising, but possible)
- **Mul never grokks** → three tasks may exceed the model's representational capacity at this size

Saves to `results/phase2_grokking_curves.png`

In [ ]:
COLORS = {'add': 'steelblue', 'sub': 'darkorange', 'mul': 'seagreen'}
epochs = history['epoch']

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle(f'Phase 2 — Multi-Task Grokking: Add + Sub + Mul (mod {P})',
             fontsize=14, fontweight='bold')

# Loss
ax = axes[0]
ax.semilogy(epochs, history['train_loss'], label='Train loss', color='steelblue')
ax.semilogy(epochs, history['test_loss'],  label='Test loss',  color='tomato')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss (log scale)')
ax.set_title('Loss Curves (Overall)'); ax.legend(); ax.grid(True, which='both', alpha=0.3)

# Per-task accuracy
ax = axes[1]
grok_epochs = {}
for op in OP_LIST:
    c = COLORS.get(op, 'purple')
    ax.plot(epochs, history[f'train_acc_{op}'], '--', color=c, alpha=0.4, label=f'Train [{op}]')
    ax.plot(epochs, history[f'test_acc_{op}'],  '-',  color=c,             label=f'Test  [{op}]')
    grok_ep = next((ep for ep, a in zip(epochs, history[f'test_acc_{op}']) if a > 0.95), None)
    if grok_ep is not None:
        grok_epochs[op] = grok_ep
        ax.axvline(grok_ep, linestyle=':', color=c, alpha=0.7)
        ax.text(grok_ep + 100, 0.05 + list(OP_LIST).index(op) * 0.08,
                f'{op}@{grok_ep}', color=c, fontsize=8)

ax.set_xlabel('Epoch'); ax.set_ylabel('Test Accuracy')
ax.set_title('Per-Task Accuracy — Phase Transitions')
ax.set_ylim(-0.05, 1.08); ax.legend(loc='upper left', fontsize=8); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/phase2_grokking_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved results/phase2_grokking_curves.png')

print('\nGrokking epochs:')
for op in OP_LIST:
    ep = grok_epochs.get(op, 'DID NOT GROK')
    print(f'  {op}: {ep}')

# Key diagnostic: did mul grok at a different time than add/sub?
if 'mul' in grok_epochs and 'add' in grok_epochs:
    diff = abs(grok_epochs['mul'] - grok_epochs['add'])
    if diff > 2000:
        print(f'\nKey finding: mul grokked {diff} epochs LATER than add/sub')
        print('  -> Evidence of SEPARATE learning dynamics for mul')
    elif diff < 500:
        print(f'\nKey finding: mul grokked within {diff} epochs of add/sub')
        print('  -> Evidence of SHARED learning dynamics (surprising for mul!)')

## Cell 9 — Fourier Analysis

**Cross-phase comparison:** Compare dominant frequencies to Phase 0 (1 task) and Phase 1 (2 tasks).

| Phase | Tasks | Expected dominant freqs |
|-------|-------|------------------------|
| 0 | add only | ~4 spikes |
| 1 | add + sub | ~5 spikes (slight expansion) |
| 2 | add + sub + mul | ??? — more expansion if mul needs different freqs |

Saves to `results/phase2_fourier_spectrum.png`

In [ ]:
def fourier_embedding_analysis(model, p):
    W_E = model.W_E[:p].detach().cpu().numpy()
    power = (np.abs(np.fft.rfft(W_E, axis=0)) ** 2).sum(axis=1)
    return power


power = fourier_embedding_analysis(model, P)
freqs = np.arange(len(power))
top_k = 8
top_idx = np.argsort(power)[-top_k:][::-1]

fig, ax = plt.subplots(figsize=(13, 4))
ax.bar(freqs, power, color='steelblue', edgecolor='white', linewidth=0.3)
for idx in top_idx:
    ax.bar(idx, power[idx], color='crimson')
    ax.annotate(f'f={idx}', xy=(idx, power[idx]),
                xytext=(idx + 0.8, power[idx] * 1.04),
                fontsize=7.5, color='crimson',
                arrowprops=dict(arrowstyle='->', color='crimson', lw=0.7))

# Phase comparison annotation
phase1_freqs = [9, 18, 28, 47, 49]  # from your Phase 1 run
ax.set_xlabel('Frequency')
ax.set_ylabel('Power (sum over d_model)')
ax.set_title(
    f'Phase 2 — Fourier Spectrum (mod {P}, add+sub+mul)\n'
    f'Top-{top_k} freqs in red | Phase 1 top freqs were: {phase1_freqs}'
)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('results/phase2_fourier_spectrum.png', dpi=150, bbox_inches='tight')
plt.show()

top_nondc = sorted([f for f in top_idx if f > 0])
print(f'Top non-DC frequencies: {top_nondc}')
print(f'Phase 1 top non-DC:     {phase1_freqs}')

overlap = set(top_nondc) & set(phase1_freqs)
new_freqs = set(top_nondc) - set(phase1_freqs)
print(f'Shared with Phase 1: {sorted(overlap)}')
print(f'New frequencies:     {sorted(new_freqs)}')
if new_freqs:
    print('  -> New frequencies suggest the model expanded its Fourier basis for mul')
else:
    print('  -> Same frequencies as Phase 1 -> circuit did NOT expand for mul')

## Cell 10 — Head Ablation Sweep (3-task, fixed)

**Fix from Phase 1:** The `use_attn_result=True` flag in the model config means `hook_result` is now actually computed and can be zeroed. Phase 1 showed all-zero drops because the hook existed as a name but wasn't materialised.

**What to look for (3-task version):**

| Pattern | Interpretation |
|---------|---------------|
| Head drops all 3 tasks | Fully shared across all operations |
| Head drops add+sub only, mul unaffected | Shared add/sub head; mul uses separate circuit |
| Head drops mul only | Mul-specific head |
| Head drops nothing | Not critical for any task |

Saves to `results/phase2_head_ablation.png`

In [ ]:
@torch.no_grad()
def ablate_head_accuracy(model, inputs, labels, op_tokens_map,
                          layer=0, head_idx=0, device=DEVICE):
    """Zero out head_idx in layer, measure per-task accuracy."""
    model.eval()
    inputs, labels = inputs.to(device), labels.to(device)

    # hook_result shape: [batch, seq, n_heads, d_head]
    # Only works when use_attn_result=True in the model config
    hook_name = f'blocks.{layer}.attn.hook_result'

    def zero_head(activation, hook):
        act = activation.clone()
        act[:, :, head_idx, :] = 0.0
        return act

    logits = model.run_with_hooks(inputs, fwd_hooks=[(hook_name, zero_head)])[:, -1, :]
    preds  = logits.argmax(-1)
    return {
        op: (preds[inputs[:, 1] == tok] == labels[inputs[:, 1] == tok]).float().mean().item()
        if (inputs[:, 1] == tok).sum() > 0 else float('nan')
        for op, tok in op_tokens_map.items()
    }


# Baseline
baseline = per_task_accuracy(model, test_inputs, test_labels, OP_TOKENS)
print('Baseline (no ablation):')
for op, acc in baseline.items(): print(f'  {op}: {acc:.4f}')

# Ablation sweep
ablation = {}
for h in range(N_HEADS):
    ablated = ablate_head_accuracy(model, test_inputs, test_labels, OP_TOKENS, head_idx=h)
    ablation[h] = {op: baseline[op] - ablated[op] for op in OP_LIST}
    drops_str = '  '.join(f'{op}: {ablation[h][op]:+.4f}' for op in OP_LIST)
    print(f'  Head {h}: {drops_str}')

print('\nAblation sweep done.')

In [ ]:
# Plot ablation results
COLORS = {'add': 'steelblue', 'sub': 'darkorange', 'mul': 'seagreen'}
x = np.arange(N_HEADS)
bar_w = 0.25
THRESHOLD = 0.10

fig, ax = plt.subplots(figsize=(11, 5))
for i, op in enumerate(OP_LIST):
    drops = [ablation[h][op] for h in range(N_HEADS)]
    offset = (i - len(OP_LIST)/2 + 0.5) * bar_w
    bars = ax.bar(x + offset, drops, bar_w,
                  label=f'drop [{op}]', color=COLORS[op], edgecolor='white')
    for bar, val in zip(bars, drops):
        if not math.isnan(val):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                    f'{val:.2f}', ha='center', va='bottom', fontsize=7.5)

ax.axhline(THRESHOLD, color='red', linestyle='--', linewidth=0.8, label='10% threshold')
ax.axhline(0, color='black', linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels([f'Head {h}' for h in range(N_HEADS)])
ax.set_ylabel('Accuracy drop when head is zeroed')
ax.set_title(
    f'Phase 2 — Head Ablation: Add vs Sub vs Mul (mod {P})\n'
    'Both add+sub drop but not mul -> mul has separate circuit\n'
    'All three drop -> fully shared head'
)
ax.legend()
ax.grid(True, axis='y', alpha=0.3)
ax.set_ylim(-0.05, max(0.5, max(
    ablation[h][op] for h in range(N_HEADS) for op in OP_LIST
) + 0.15))

plt.tight_layout()
plt.savefig('results/phase2_head_ablation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved results/phase2_head_ablation.png')

print('\nInterpretation (threshold = 10% drop):')
for h in range(N_HEADS):
    critical = [op for op in OP_LIST if ablation[h][op] >= THRESHOLD]
    if len(critical) == 3:   label = 'SHARED (all 3 tasks)'
    elif len(critical) == 2: label = f'PARTIAL — critical for {critical}'
    elif len(critical) == 1: label = f'TASK-SPECIFIC [{critical[0]}]'
    else:                    label = 'NOT CRITICAL'
    print(f'  Head {h}: {label}')

## Cell 11 — Pairwise Activation Patching (6 directed pairs)

Phase 1 tested only add↔sub. Phase 2 tests all 3 directed pairs:
- **add → sub** and **sub → add** (should have HIGH overlap, as in Phase 1)
- **add → mul** and **mul → add** (expect LOWER overlap — mul is different)
- **sub → mul** and **mul → sub** (expect LOWER overlap)

For each pair, we patch at three hook points: token embedding, attention output, MLP output.

Saves to `results/phase2_activation_patching.png`

In [ ]:
FIXED_B = 7  # Fix b=7, sweep all a values

# Build per-operation matched inputs (same a values, same b, different op token)
op_inputs  = {}
op_labels  = {}
OP_FNS = {'add': lambda a: (a + FIXED_B) % P,
           'sub': lambda a: (a - FIXED_B) % P,
           'mul': lambda a: (a * FIXED_B) % P}

for op in OP_LIST:
    tok = OP_TOKENS[op]
    op_inputs[op] = torch.tensor(
        [[a, tok, FIXED_B, EQUALS_TOKEN] for a in range(P)], dtype=torch.long
    ).to(DEVICE)
    op_labels[op] = torch.tensor(
        [OP_FNS[op](a) for a in range(P)], dtype=torch.long
    ).to(DEVICE)

@torch.no_grad()
def patch_accuracy(model, clean_op, corrupted_op, hook_name, device=DEVICE):
    """Patch clean_op's activations at hook_name into corrupted_op's forward pass.
    Returns accuracy on the corrupted_op's labels.
    Low accuracy -> hook carries operation-specific info that was overwritten.
    Near-baseline accuracy -> hook is shared / not the critical component.
    """
    clean_in  = op_inputs[clean_op]
    corr_in   = op_inputs[corrupted_op]
    corr_lbl  = op_labels[corrupted_op]

    try:
        _, clean_cache = model.run_with_cache(clean_in)
        clean_act = clean_cache[hook_name]

        def patch_fn(act, hook):
            out = act.clone()
            b = min(out.shape[0], clean_act.shape[0])
            out[:b] = clean_act[:b]
            return out

        logits = model.run_with_hooks(corr_in, fwd_hooks=[(hook_name, patch_fn)])[:, -1, :]
        return (logits.argmax(-1) == corr_lbl).float().mean().item()
    except Exception as e:
        print(f'  Hook {hook_name} failed ({clean_op}->{corrupted_op}): {e}')
        return float('nan')


HOOK_POINTS = [
    ('hook_embed',              'Token Embed'),
    ('blocks.0.attn.hook_result', 'Attn Output'),
    ('blocks.0.hook_mlp_out',   'MLP Output'),
]

# Compute baselines
baselines = {}
for op in OP_LIST:
    model.eval()
    with torch.no_grad():
        logits = model(op_inputs[op])[:, -1, :]
        baselines[op] = (logits.argmax(-1) == op_labels[op]).float().mean().item()
print('Baselines (b=7):', {op: f'{v:.4f}' for op, v in baselines.items()})

# All directed pairs
PAIRS = [(a, b) for a in OP_LIST for b in OP_LIST if a != b]

patch_results = {}  # (clean, corr, hook) -> acc
for clean, corr in PAIRS:
    for hook_name, hook_label in HOOK_POINTS:
        acc = patch_accuracy(model, clean, corr, hook_name)
        patch_results[(clean, corr, hook_label)] = acc

print('\nActivation patching done.')

In [ ]:
# Plot: 3-panel (one per hook point), x-axis = directed pair, y = accuracy
hook_labels = [hl for _, hl in HOOK_POINTS]
pair_labels = [f'{c}\u2192{r}' for c, r in PAIRS]  # 'add->sub', etc.

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
fig.suptitle(
    f'Phase 2 — Activation Patching: All Pairs (mod {P}, b={FIXED_B})\n'
    'Near-baseline = shared component; near-zero = operation-specific component',
    fontsize=12, fontweight='bold'
)

PAIR_COLORS = {'add': 'steelblue', 'sub': 'darkorange', 'mul': 'seagreen'}

for ax, (hook_name, hook_label) in zip(axes, HOOK_POINTS):
    accs   = [patch_results[(c, r, hook_label)] for c, r in PAIRS]
    colors = [PAIR_COLORS[c] for c, r in PAIRS]

    bars = ax.bar(range(len(PAIRS)), accs, color=colors, edgecolor='white', alpha=0.85)

    # Baseline dotted lines per target op
    for op in OP_LIST:
        ax.axhline(baselines[op], color=PAIR_COLORS[op], linestyle=':', linewidth=1,
                   alpha=0.6, label=f'{op} baseline ({baselines[op]:.2f})')

    for bar, val in zip(bars, accs):
        if not math.isnan(val):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f'{val:.2f}', ha='center', va='bottom', fontsize=7.5)

    ax.set_xticks(range(len(PAIRS)))
    ax.set_xticklabels(pair_labels, rotation=35, ha='right', fontsize=8)
    ax.set_title(hook_label)
    ax.set_ylim(0, 1.2)
    ax.grid(True, axis='y', alpha=0.3)
    if ax == axes[0]:
        ax.set_ylabel('Accuracy after patching')
    ax.legend(fontsize=6, loc='upper right')

plt.tight_layout()
plt.savefig('results/phase2_activation_patching.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved results/phase2_activation_patching.png')

# Summary: compare add<->sub overlap vs add<->mul and sub<->mul
print('\nPairwise overlap comparison (Token Embed hook):')
hl = 'Token Embed'
add_sub_avg = (patch_results[('add','sub',hl)] + patch_results[('sub','add',hl)]) / 2
add_mul_avg = (patch_results[('add','mul',hl)] + patch_results[('mul','add',hl)]) / 2
sub_mul_avg = (patch_results[('sub','mul',hl)] + patch_results[('mul','sub',hl)]) / 2
print(f'  add <-> sub: {add_sub_avg:.4f}')
print(f'  add <-> mul: {add_mul_avg:.4f}')
print(f'  sub <-> mul: {sub_mul_avg:.4f}')
if add_mul_avg > add_sub_avg - 0.05:
    print('  -> mul shares as much as add/sub: unexpectedly high sharing!')
elif add_mul_avg < 0.5:
    print('  -> mul is significantly more isolated than add/sub')

## Cell 12 — Cross-Phase Summary

Prints a side-by-side comparison of Phase 1 and Phase 2 findings — this is what you copy into your research log and `docs/proposal.md`.

In [ ]:
print('=' * 72)
print('CROSS-PHASE COMPARISON: Phase 1 (add+sub) vs Phase 2 (add+sub+mul)')
print('=' * 72)

print('\n1. GROKKING')
print('   Phase 1: both add+sub grokked at epoch 6200 (same time -> shared dynamics)')
print('   Phase 2:')
for op in OP_LIST:
    acc = history[f'test_acc_{op}'][-1]
    ep = grok_epochs.get(op, 'DID NOT GROK')
    print(f'     {op}: acc={acc:.4f}, grok epoch={ep}')

print('\n2. FOURIER SPECTRUM')
print(f'   Phase 1: {{9, 18, 28, 47, 49}} (5 non-DC peaks)')
print(f'   Phase 2: {set(top_nondc)} ({len(top_nondc)} non-DC peaks)')
if len(top_nondc) > 5:
    print('   -> More frequencies needed: mul expanded the Fourier basis')
elif len(top_nondc) <= 5:
    print('   -> Same frequency count: mul did not expand the basis (or uses same freqs)')

print('\n3. HEAD ABLATION')
print('   Phase 1: all heads showed 0 drop (use_attn_result bug -> results invalid)')
print('   Phase 2 (fixed):')
for h in range(N_HEADS):
    critical = [op for op in OP_LIST if ablation[h][op] >= THRESHOLD]
    print(f'     Head {h}: critical for {critical if critical else "none"}')

print('\n4. ACTIVATION PATCHING (Token Embed hook)')
print('   Phase 1: add->sub = 0.0, sub->add = 0.0  (token embed encodes op-specific info)')
print('   Phase 2:')
for c, r in [('add','sub'),('add','mul'),('sub','mul')]:
    a2b = patch_results[(c, r, 'Token Embed')]
    b2a = patch_results[(r, c, 'Token Embed')]
    print(f'     {c}<->{r}: {c}->{r}={a2b:.4f}, {r}->{c}={b2a:.4f}')

print('\n5. SAVED FILES')
for f in ['results/phase2_grokking_curves.png', 'results/phase2_fourier_spectrum.png',
          'results/phase2_head_ablation.png',   'results/phase2_activation_patching.png']:
    status = 'OK' if os.path.exists(f) else 'MISSING'
    print(f'   [{status}] {f}')

print('\n' + '=' * 72)
print('Next: download all 4 PNGs from Colab, commit to results/, update README')
print('Then start Phase 3: add a non-abelian operation (permutation composition S5)')
print('=' * 72)

## Cell 13 — (Optional) Save to Google Drive

In [ ]:
SAVE_TO_DRIVE = False  # Set True to persist between sessions

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    save_dir = '/content/drive/MyDrive/mech_interp_phase2'
    os.makedirs(save_dir, exist_ok=True)
    torch.save(model.state_dict(), f'{save_dir}/model_final.pt')
    import json
    with open(f'{save_dir}/history.json', 'w') as f:
        json.dump(history, f)
    import shutil
    for png in ['phase2_grokking_curves.png','phase2_fourier_spectrum.png',
                'phase2_head_ablation.png','phase2_activation_patching.png']:
        shutil.copy(f'results/{png}', f'{save_dir}/{png}')
    print(f'Saved model, history, and 4 PNGs to {save_dir}')
else:
    print('Drive save skipped. Set SAVE_TO_DRIVE=True to persist across sessions.')
    print('Remember to download the 4 PNGs from the Colab file browser before session ends!')